In [0]:
# 00_utils — shared helpers for the NYC taxi medallion pipeline
# Imported into every stage notebook via: %run ./00_utils

from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

LOG_TABLE = "workspace.taxi.pipeline_log"
PIPELINE_NAME = "nyc_taxi_yellow"

LOG_SCHEMA = StructType([
    StructField("pipeline_name", StringType(),    nullable=False),
    StructField("run_id",        StringType(),    nullable=False),
    StructField("stage",         StringType(),    nullable=False),
    StructField("rows_in",       LongType(),      nullable=True),
    StructField("rows_out",      LongType(),      nullable=True),
    StructField("status",        StringType(),    nullable=False),
    StructField("error_message", StringType(),    nullable=True),
    StructField("run_timestamp", TimestampType(), nullable=False),
])

def log_pipeline_run(stage, rows_in, rows_out, status, run_id, error_message=None):
    """Append one observability row to the pipeline_log Delta table."""
    log_row = spark.createDataFrame(
        [(PIPELINE_NAME, run_id, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],
        schema=LOG_SCHEMA
    )
    (log_row.write.format("delta").mode("append").saveAsTable(LOG_TABLE))
    print(f"[{stage}] {status} | rows_in={rows_in:,} rows_out={rows_out:,}")

print("Loaded helpers: LOG_TABLE, LOG_SCHEMA, log_pipeline_run, PIPELINE_NAME")

Loaded helpers: LOG_TABLE, LOG_SCHEMA, log_pipeline_run, PIPELINE_NAME
